# 01 — EDA: Synthetic Org Tree + Skills Profile

**Project H20 — Succession-Planning Graph Recommender.** 2,000-employee org tree, 12 roles across 7 levels, plus a 40-dim skill vector per employee drawn from a per-role centroid + Gaussian noise. We profile the graph and the skills before any embedding.

In [ ]:
from pathlib import Path
import sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

sns.set_theme(style='whitegrid')
df = pd.read_parquet('../data/processed/employee_attrs.parquet')
with open('../data/processed/org_graph.gpickle', 'rb') as fh:
    G = pickle.load(fh)
df['skills'] = df['skills'].apply(np.asarray)
df.head()

## 1. Org-tree shape

In [ ]:
print(f'employees: {len(df):,}')
print(f'graph: nodes={G.number_of_nodes():,}   edges={G.number_of_edges():,}')
print(f'is tree (after undirected): {nx.is_tree(G.to_undirected())}')
print(f'roles: {df["role"].nunique()}    levels: {sorted(df["level"].unique())}')

## 2. Role and level distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df['role'].value_counts().plot.barh(ax=axes[0], color='#1f77b4')
axes[0].set_title('Headcount per role')
df['level'].value_counts().sort_index().plot.bar(ax=axes[1], color='#9467bd')
axes[1].set_title('Headcount per level (1=IC, 7=CEO)')
plt.tight_layout(); plt.show()

## 3. Span of control distribution

In [ ]:
out_deg = pd.Series([G.out_degree(n) for n in G.nodes()])
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(out_deg, bins=20, color='#2ca02c', ax=ax)
ax.set_xlabel('direct reports'); ax.set_title('Span of control (out-degree)')
plt.tight_layout(); plt.show()
print(out_deg.describe())

## 4. Tenure and performance distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['tenure_yrs'], bins=30, color='#1f77b4', ax=axes[0])
axes[0].set_title('Tenure (years)')
sns.countplot(data=df, x='performance_rating', palette='Set2', ax=axes[1])
axes[1].set_title('Performance rating')
plt.tight_layout(); plt.show()

## 5. Skills matrix shape and norm

In [ ]:
S = np.array([np.asarray(v) for v in df['skills']])
print(f'skills matrix: {S.shape}')
norms = np.linalg.norm(S, axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(norms, bins=30, color='#9467bd', ax=ax)
ax.set_title('Per-employee skill-vector L2 norm')
plt.tight_layout(); plt.show()

## 6. Role centroid heatmap (mean skill vector per role)

In [ ]:
df['skill_arr'] = df['skills'].apply(np.asarray)
role_means = df.groupby('role').apply(lambda g: np.stack(g['skill_arr']).mean(axis=0))
M = np.vstack(role_means.values)
fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(M, ax=ax, cmap='RdBu_r', center=0,
            yticklabels=role_means.index, xticklabels=False, cbar_kws={'label': 'mean skill weight'})
ax.set_title('Per-role mean skill vector (40 dims)')
plt.tight_layout(); plt.show()

## 7. Pairwise role centroid cosine

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
C = cosine_similarity(M)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(C, annot=True, fmt='.2f', xticklabels=role_means.index, yticklabels=role_means.index,
            cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Role-centroid pairwise cosine')
plt.tight_layout(); plt.show()

## 8. Subordinate counts of leadership roles

In [ ]:
leaders = df[df['role'].isin(['CEO', 'VP Engineering', 'VP Sales', 'VP HR',
                                'Director Eng', 'Director Sales', 'Director Ops', 'Director Marketing'])]
rows = []
for eid in leaders['emp_id']:
    descendants = nx.descendants(G, eid)
    rows.append(dict(emp_id=eid, role=df[df['emp_id']==eid]['role'].iloc[0],
                      total_subordinates=len(descendants)))
lead_df = pd.DataFrame(rows).sort_values('total_subordinates', ascending=False)
print(lead_df.head(10))

## 9. Implications for the recommender
- Skills are clearly clustered by role (centroid heatmap + cosine block-diagonal).
- The org tree has typical leadership layers and span-of-control distributions.
- Manager nodes have substantial descendant pools — succession candidates can be drawn from outside the immediate team via the spectral embedding.